# Feature experiment — estimating per-paper TRL from abstract text

**ELI5:** `trl_min`/`trl_max` in `papers_combined.parquet` are broadcast constants
per use case (the analyst's stated acceptable TRL window for the whole research
question) — every paper in a use case carries the *same* two numbers, so they carry
zero per-paper signal. This notebook is a small, honest viability check for a
genuinely new row-level feature: **can we estimate an individual paper's own TRL
from its title + abstract text?**

This is NOT the full Week-3 pipeline — it's a ~30-paper spot check to decide
whether the idea is worth building out further. No new heavy dependencies: plain
Python string/keyword matching, checked against my own hand-read judgement on the
same sample.

In [1]:
import pandas as pd

DATA_PATH = "../../data/processed/papers_combined.parquet"

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns from {DATA_PATH}")

Loaded 2873 rows, 50 columns from ../../data/processed/papers_combined.parquet


## 1. Sample selection

Not `df.sample()` — I read ~90 candidate abstracts (positively-triaged, `has_abstract`)
spread across all 6 use cases and hand-picked 34 that span a real spread of maturity
cues: pure simulation/algorithm papers, lab/bench experiments, greenhouse and
pilot-plant trials, and a few genuine late-stage cases (a real ISS flight test, a
review of an already-commercialised binder technology, a field trial with quantified
crop yields). A few "keyword trap" papers are included on purpose (e.g. a title
containing *"Industrial-Grade"* about a lab-record device, not a deployed product) —
those are exactly the cases a naive keyword heuristic should be expected to get wrong.

In [2]:
SAMPLE_IDS = [
    # carbon_capture
    "carbon_capture__3542", "carbon_capture__3352", "carbon_capture__1142",
    "carbon_capture__1613", "carbon_capture__1152", "carbon_capture__1626",
    # cement_binders
    "cement_binders__255", "cement_binders__657", "cement_binders__201",
    "cement_binders__512", "cement_binders__439", "cement_binders__368",
    # ner
    "ner__2248", "ner__2958", "ner__2758", "ner__2522", "ner__2145",
    # soil_microbiome
    "soil_microbiome__361", "soil_microbiome__484", "soil_microbiome__245",
    "soil_microbiome__491", "soil_microbiome__698",
    # solar_leo
    "solar_leo__2020", "solar_leo__1909", "solar_leo__3076", "solar_leo__2013",
    "solar_leo__3055", "solar_leo__1952", "solar_leo__3001",
    # tech_forecasting
    "tech_forecasting__1033", "tech_forecasting__3184", "tech_forecasting__1107",
    "tech_forecasting__3141", "tech_forecasting__722",
]

sample = df[df["paper_id"].isin(SAMPLE_IDS)].set_index("paper_id").loc[SAMPLE_IDS].reset_index()
assert len(sample) == len(SAMPLE_IDS), "one or more sample paper_ids not found in the export"
assert sample["has_abstract"].all(), "sample must only contain papers with an abstract"

print(f"Sample size: {len(sample)} papers across {sample['use_case_key'].nunique()} use cases")
sample["use_case_key"].value_counts()

Sample size: 34 papers across 6 use cases


use_case_key
solar_leo           7
carbon_capture      6
cement_binders      6
ner                 5
soil_microbiome     5
tech_forecasting    5
Name: count, dtype: int64

In [3]:
sample[["paper_id", "use_case_key", "year", "triage_label", "trl_min", "trl_max"]]

,paper_id,use_case_key,year,triage_label,trl_min,trl_max
0,carbon_capture__3542,carbon_capture,2022,positive,<NA>,<NA>
1,carbon_capture__3352,carbon_capture,2022,positive,<NA>,<NA>
2,carbon_capture__1142,carbon_capture,2025,positive,<NA>,<NA>
3,carbon_capture__1613,carbon_capture,2019,positive,<NA>,<NA>
4,carbon_capture__1152,carbon_capture,2024,positive,<NA>,<NA>
5,carbon_capture__1626,carbon_capture,2022,positive,<NA>,<NA>
6,cement_binders__255,cement_binders,2025,positive,1,4
7,cement_binders__657,cement_binders,2020,positive,1,4
8,cement_binders__201,cement_binders,2025,positive,1,4
9,cement_binders__512,cement_binders,2023,positive,1,4


## 2. Hand-labelled ground truth

For each of the 34 papers I read the title + abstract myself and assigned my own
best-judgement TRL: a coarse band (`early` 1-3 / `mid` 4-6 / `late` 7-9) plus a
specific number, with a one-line reason. **Ground rule used throughout:** the label
reflects the maturity of the *technology or finding described*, not the novelty of
the paper's own methodological contribution — e.g. a paper that *analyses* an
existing pilot plant's operational data is graded on the pilot plant's maturity
(mid/late), not on how novel the analysis method is.

This is a judgement call, not a computed fact — it's the "critic" this notebook
checks the keyword heuristic against, and it's honest precisely because it's my own
read rather than a fabricated number.

In [4]:
HAND_LABELS = {
    "carbon_capture__3542": ("early", 2, "Purely in-silico solvent screening (kinetics + MD + ML) - no physical experiment"),
    "carbon_capture__3352": ("early", 2, "A conceptual 'pre-assessment' of a proposed micro-turbine CCS integration, no experimental data"),
    "carbon_capture__1142": ("mid", 4, "Novel two-stage electrochemical system built and tested at bench scale for the first time"),
    "carbon_capture__1613": ("mid", 5, "Bench-scale pilot-plant experiments benchmarking a new solvent's regeneration energy against MEA"),
    "carbon_capture__1152": ("mid", 4, "Review synthesising lab-stage amine/catalyst methods; 'promote widespread adoption' outruns the evidence cited"),
    "carbon_capture__1626": ("mid", 6, "Mechanistic/simulation analysis of data from a real long-term pilot-plant operation"),

    "cement_binders__255": ("early", 2, "Pure ML/optimisation framework fit to an existing 387-specimen dataset - no new material made"),
    "cement_binders__657": ("early", 3, "Lab paste-sample characterisation (strength, microscopy) under controlled heating"),
    "cement_binders__201": ("mid", 4, "Lab feasibility mix using real local materials (Burkina Faso), reports an achieved strength gain"),
    "cement_binders__512": ("late", 9, "Review states outright the alkali-activated technologies are 'already available in the industry'"),
    "cement_binders__439": ("mid", 4, "LCA of an existing byproduct-based binder using literature/production data, not new lab synthesis"),
    "cement_binders__368": ("mid", 6, "Real pilot-plant castings (60 kg) at an industrial foundry partner under an EU LIFE project"),

    "ner__2248": ("early", 2, "New label-noise-correction method for entity typing, tested on academic benchmarks only"),
    "ner__2958": ("early", 2, "New entity-relation extraction model (MoE + dependency parsing), academic benchmarks only"),
    "ner__2758": ("mid", 4, "Applies 5 transformer models to real government/news/TripAdvisor text, not a lab-only benchmark"),
    "ner__2522": ("mid", 4, "Stress-tests existing production-grade NER models (ParsBERT, XLM-R) against real user-generated noise"),
    "ner__2145": ("early", 3, "Builds a new biomedical NER corpus/resource - a foundational resource, not an applied system"),

    "soil_microbiome__361": ("early", 2, "Opinion article discussing others' work; presents no new primary data"),
    "soil_microbiome__484": ("early", 2, "Conceptual/paradigm-shift review proposing a research direction, no new experiment"),
    "soil_microbiome__245": ("mid", 4, "Greenhouse SynCom experiment in realistic non-sterile soil - lab/greenhouse validation"),
    "soil_microbiome__491": ("mid", 4, "Greenhouse SynCom experiment with quantified tomato growth response"),
    "soil_microbiome__698": ("late", 7, "Real field trials (not greenhouse) with quantified cotton yield/height gains in Kazakhstan"),

    "solar_leo__2020": ("early", 2, "Pure opto-electro-thermal simulation model - no device fabricated"),
    "solar_leo__1909": ("early", 2, "Short, aspirational statement about space potential - no experimental data given"),
    "solar_leo__3076": ("early", 3, "Lab-fabricated device with low/uncompetitive efficiency (10.79% PCE) - early bench stage"),
    "solar_leo__2013": ("mid", 4, "Lab champion cell, certified efficiency, but still a small-area R&D device"),
    "solar_leo__3055": ("mid", 5, "Explicit scale-up (1cm2 to 16cm2) plus accelerated durability testing (IEC 61215)"),
    "solar_leo__1952": ("mid", 4, "'Industrial-Grade' in the title, but content is lab-record efficiencies; adoption is a stated future goal - a keyword trap"),
    "solar_leo__3001": ("late", 7, "Real flight test aboard the ISS (MISSE mission) - a genuine operational (orbital) environment"),

    "tech_forecasting__1033": ("early", 2, "New link-prediction algorithm (DANOM), tested on academic benchmark datasets"),
    "tech_forecasting__3184": ("early", 2, "Survey of link-prediction algorithms - no new implementation or deployment"),
    "tech_forecasting__1107": ("mid", 4, "Applies a KG-mining model to a real patent database, reports a quantified coverage metric"),
    "tech_forecasting__3141": ("mid", 4, "Large-scale empirical analysis of 33,881 real patent descriptions"),
    "tech_forecasting__722": ("early", 3, "Review of computational drug-repurposing methods - the methods reviewed are still research-stage tools"),
}

hand_df = pd.DataFrame(
    [(pid, *v) for pid, v in HAND_LABELS.items()],
    columns=["paper_id", "hand_band", "hand_trl", "hand_reason"],
)
sample = sample.merge(hand_df, on="paper_id", validate="one_to_one")
sample["hand_band"].value_counts().rename_axis("band").to_frame("n_papers")

,n_papers
band,
mid,16
early,15
late,3


In [5]:
sample[["paper_id", "use_case_key", "title", "hand_band", "hand_trl", "hand_reason"]]

,paper_id,use_case_key,title,hand_band,hand_trl,hand_reason
0,carbon_capture__3542,carbon_capture,Computational screening methodology identifies...,early,2,Purely in-silico solvent screening (kinetics +...
1,carbon_capture__3352,carbon_capture,Absorption-based carbon capture energy penalty...,early,2,A conceptual 'pre-assessment' of a proposed mi...
2,carbon_capture__1142,carbon_capture,A Membraneless Electrochemically Mediated Amin...,mid,4,Novel two-stage electrochemical system built a...
3,carbon_capture__1613,carbon_capture,An experimental based optimization of a novel ...,mid,5,Bench-scale pilot-plant experiments benchmarki...
4,carbon_capture__1152,carbon_capture,Comparative Review for Enhancing CO2 Capture E...,mid,4,Review synthesising lab-stage amine/catalyst m...
5,carbon_capture__1626,carbon_capture,Mechanistic analysis of post-combustion CO2 ca...,mid,6,Mechanistic/simulation analysis of data from a...
6,cement_binders__255,cement_binders,A hybrid prediction and multi-objective optimi...,early,2,Pure ML/optimisation framework fit to an exist...
7,cement_binders__657,cement_binders,Macro–meso–micro experimental studies of calci...,early,3,"Lab paste-sample characterisation (strength, m..."
8,cement_binders__201,cement_binders,A feasibility assessment of low-carbon concret...,mid,4,Lab feasibility mix using real local materials...
9,cement_binders__512,cement_binders,A review: Alkali-activated cement and concrete...,late,9,Review states outright the alkali-activated te...


## 3. Keyword-based TRL heuristic

A transparent, three-family keyword scorer over `title + ". " + abstract`:

- **early** cues — theoretical/simulation/conceptual language (`in silico`, `simulated`,
  `hypothetical`, `preliminary`, ...)
- **mid** cues — lab/prototype/pilot/applied language (`pilot plant`, `bench-scale`,
  `greenhouse`, `feasibility`, `scale-up`, ...)
- **late** cues — deployed/commercial/operational language (`commercially available`,
  `industrial-scale`, `field trial`, `international space station`, ...)

Each family is counted (presence of a phrase, not weighted by repeat count); the band
with the most hits wins, with **late > mid > early** as the tie-break — real deployment
language is treated as a stronger, rarer signal than routine simulation wording that
shows up in almost every paper's framing. If nothing matches at all, the heuristic
defaults to `early` (no maturity/deployment language present is itself a signal of a
purely theoretical/methodological paper). This default was the one fix made after a
first pass defaulted to `mid` and did clearly worse — no other tuning was done against
the hand labels below, to keep the reported agreement honest.

In [6]:
EARLY_KW = [
    "theoretical", "conceptual", "in silico", "computational screening", "simulation",
    "simulated", "simulate", "modelled", "modeled", "proof of concept", "preliminary",
    "hypothesis", "hypothetical", "opinion article", "perspective",
]
MID_KW = [
    "prototype", "pilot plant", "pilot-scale", "pilot scale", "bench-scale", "bench scale",
    "laboratory scale", "lab-scale", "greenhouse", "demonstrated", "demonstration",
    "feasibility", "scale-up", "scaled up", "applied to", "case study",
]
LATE_KW = [
    "commercial", "commercially available", "industrial-scale", "industrial scale",
    "industrialization", "deployed", "deployment", "in production", "full-scale",
    "already available in the industry", "widespread adoption", "field trial",
    "field-scale", "market-ready", "international space station", "in orbit",
    "on-orbit", "flight test", "operational environment",
]

def keyword_trl_band(text: str) -> tuple[str, tuple[int, int, int]]:
    """Score title+abstract text against the three keyword families and return
    (predicted_band, (early_hits, mid_hits, late_hits))."""
    t = text.lower()
    e = sum(1 for k in EARLY_KW if k in t)
    m = sum(1 for k in MID_KW if k in t)
    l = sum(1 for k in LATE_KW if k in t)
    if l > 0 and l >= m and l >= e:
        band = "late"
    elif m > 0 and m >= e:
        band = "mid"
    elif e > 0:
        band = "early"
    else:
        band = "early"  # no maturity cues at all -> most conservative reading
    return band, (e, m, l)

BAND_MIDPOINT = {"early": 2, "mid": 5, "late": 8}

preds = sample.apply(
    lambda r: keyword_trl_band(f"{r['title']}. {r['abstract']}"), axis=1
)
sample["heuristic_band"] = preds.apply(lambda p: p[0])
sample["heuristic_hits"] = preds.apply(lambda p: p[1])
sample["heuristic_trl_point"] = sample["heuristic_band"].map(BAND_MIDPOINT)
sample[["paper_id", "hand_band", "heuristic_band", "heuristic_hits"]]

,paper_id,hand_band,heuristic_band,heuristic_hits
0,carbon_capture__3542,early,early,"(3, 0, 1)"
1,carbon_capture__3352,early,late,"(1, 1, 1)"
2,carbon_capture__1142,mid,mid,"(0, 2, 0)"
3,carbon_capture__1613,mid,mid,"(0, 2, 0)"
4,carbon_capture__1152,mid,late,"(0, 0, 1)"
5,carbon_capture__1626,mid,early,"(2, 1, 0)"
6,cement_binders__255,early,mid,"(0, 1, 0)"
7,cement_binders__657,early,early,"(0, 0, 0)"
8,cement_binders__201,mid,mid,"(0, 1, 0)"
9,cement_binders__512,late,late,"(0, 1, 1)"


## 4. Heuristic vs. hand-label agreement

The honest number: does the keyword heuristic's band match my own hand-read band on
this sample? Compared against a naive baseline of always guessing the sample's
majority band, so "better than chance" has a real reference point.

In [7]:
sample["agree"] = sample["hand_band"] == sample["heuristic_band"]

n = len(sample)
n_agree = int(sample["agree"].sum())
agreement_rate = n_agree / n

majority_band = sample["hand_band"].value_counts().idxmax()
majority_baseline = (sample["hand_band"] == majority_band).mean()

print(f"Heuristic agreement with hand labels: {n_agree}/{n} = {agreement_rate:.1%}")
print(f"Naive majority-band baseline ('{majority_band}' every time): {majority_baseline:.1%}")

Heuristic agreement with hand labels: 18/34 = 52.9%
Naive majority-band baseline ('mid' every time): 47.1%


In [8]:
confusion = pd.crosstab(sample["hand_band"], sample["heuristic_band"],
                         rownames=["hand label"], colnames=["heuristic"])
confusion = confusion.reindex(index=["early", "mid", "late"], columns=["early", "mid", "late"], fill_value=0)
confusion

heuristic,early,mid,late
hand label,,,
early,11,1,3
mid,10,4,2
late,0,0,3


In [9]:
sample.loc[~sample["agree"], ["paper_id", "use_case_key", "hand_band", "heuristic_band", "heuristic_hits", "hand_reason"]]

,paper_id,use_case_key,hand_band,heuristic_band,heuristic_hits,hand_reason
1,carbon_capture__3352,carbon_capture,early,late,"(1, 1, 1)",A conceptual 'pre-assessment' of a proposed mi...
4,carbon_capture__1152,carbon_capture,mid,late,"(0, 0, 1)",Review synthesising lab-stage amine/catalyst m...
5,carbon_capture__1626,carbon_capture,mid,early,"(2, 1, 0)",Mechanistic/simulation analysis of data from a...
6,cement_binders__255,cement_binders,early,mid,"(0, 1, 0)",Pure ML/optimisation framework fit to an exist...
10,cement_binders__439,cement_binders,mid,early,"(0, 0, 0)",LCA of an existing byproduct-based binder usin...
14,ner__2758,ner,mid,early,"(0, 0, 0)",Applies 5 transformer models to real governmen...
15,ner__2522,ner,mid,early,"(2, 0, 0)",Stress-tests existing production-grade NER mod...
19,soil_microbiome__245,soil_microbiome,mid,early,"(0, 0, 0)",Greenhouse SynCom experiment in realistic non-...
20,soil_microbiome__491,soil_microbiome,mid,early,"(0, 0, 0)",Greenhouse SynCom experiment with quantified t...
22,solar_leo__2020,solar_leo,early,late,"(1, 0, 1)",Pure opto-electro-thermal simulation model - n...


**Finding:** the heuristic's agreement with my own hand labels is weak — only
18/34 (53%), barely ahead of the 47% a naive "always guess the majority band"
baseline gets for free on this sample. This is a hard computed number, not a
judgement call. **Judgement call, reading the disagreements:** the dominant failure
mode is domain-dependent — plain academic ML/NLP methods papers (`ner`,
`tech_forecasting`) almost never use any physical-science maturity vocabulary at all
(no "simulation", "pilot", "commercial", ...), so the heuristic has nothing to key
off there and gets several early papers right only by the early-default, not a real
signal. The materials/energy papers also expose the deliberate keyword-trap cases
built into the sample (e.g. "industrial-scale" describing someone else's existing
silicon substrate, not the paper's own result; "Industrial-Grade" in a title about a
lab-record device) — genuine, not rare, failure modes for a plain keyword list.

## 5. Cross-check against each use case's own TRL window

`trl_min`/`trl_max` only exist for 3 of the 6 use cases in this dataset
(`cement_binders` 1-4, `ner` 6-9, `solar_leo` 3-9) — the other 3 never defined one.
For the sampled papers in those 3 use cases: does my hand-judged (honest) per-paper
TRL actually fall inside that use case's stated window? This doesn't validate the
heuristic itself — it's a separate, interesting check of whether the *window* would
even be a useful per-paper filter if a real per-paper TRL existed.

In [10]:
has_window = sample.dropna(subset=["trl_min", "trl_max"]).copy()
has_window["hand_in_window"] = (
    (has_window["hand_trl"] >= has_window["trl_min"]) & (has_window["hand_trl"] <= has_window["trl_max"])
)
has_window["heuristic_in_window"] = (
    (has_window["heuristic_trl_point"] >= has_window["trl_min"]) & (has_window["heuristic_trl_point"] <= has_window["trl_max"])
)

cols = ["paper_id", "use_case_key", "trl_min", "trl_max", "hand_trl", "hand_in_window",
        "heuristic_trl_point", "heuristic_in_window"]
has_window[cols]

,paper_id,use_case_key,trl_min,trl_max,hand_trl,hand_in_window,heuristic_trl_point,heuristic_in_window
6,cement_binders__255,cement_binders,1,4,2,True,5,False
7,cement_binders__657,cement_binders,1,4,3,True,2,True
8,cement_binders__201,cement_binders,1,4,4,True,5,False
9,cement_binders__512,cement_binders,1,4,9,False,8,False
10,cement_binders__439,cement_binders,1,4,4,True,2,True
11,cement_binders__368,cement_binders,1,4,6,False,5,False
12,ner__2248,ner,6,9,2,False,2,False
13,ner__2958,ner,6,9,2,False,2,False
14,ner__2758,ner,6,9,4,False,2,False
15,ner__2522,ner,6,9,4,False,2,False


In [11]:
has_window.groupby("use_case_key")[["hand_in_window", "heuristic_in_window"]].agg(["sum", "count"])

hand_in_window       heuristic_in_window      
                          sum count                 sum count
use_case_key                                                 
cement_binders              4     6                   2     6
ner                         0     5                   0     5
solar_leo                   5     7                   4     7

**Finding:** `ner`'s stated window (TRL 6-9) is not matched by a single sampled
paper (0/5, hand labels or heuristic alike) — every sampled `ner` paper is academic
algorithm/methods research, nowhere near the "6-9 = demonstrated in an operational
environment" the use case's brief asks for. `cement_binders` (window 1-4, the
narrowest of the three) correctly excludes its two deliberately-included
high-maturity outliers by hand label — the already-commercial alkali-activated
review (TRL 9) and the real pilot-plant castings paper (TRL 6) both sit above the
window's cap of 4, exactly as should happen for a paper describing more mature work
than that use case asked for. `solar_leo`'s window (3-9) is wide enough that even
its outlier — the real ISS flight test (TRL 7) — sits comfortably *inside* it; that's
a different but equally sensible result, since this use case's brief evidently wants
to allow flight-tested, near-deployment work. This is a hard computed count, not a
judgement call, and it suggests the *window* concept is meaningful even though this
dataset's `trl_min`/`trl_max` columns don't vary per paper today.

## 6. Viability verdict

**Hard number:** 18/34 (53%) agreement between a lightweight keyword heuristic and my
own hand-read TRL judgement on 34 papers, barely ahead of a 47% majority-band
baseline. **Judgement call:** that is not viable as a real feature in its current
form — a plain keyword list is too easily fooled by aspirational/titular language
("Industrial-Grade", "toward industrialization") and has essentially no signal at all
for domains (NLP/ML methods papers) that don't use physical-science maturity
vocabulary in the first place.

That said, the underlying idea — a genuine row-level TRL estimate, unlike today's
constant-per-use-case columns — still looks worth pursuing, just not this way at full
scale. What the 34-paper read actually showed working: a human reader (or, at scale,
an LLM given the same "what stage is this technology at" framing used to build
Section 2's hand labels) can consistently place a paper's technology on an early/mid/
late maturity scale from its abstract alone, including catching the exact cases that
broke the keyword heuristic (title language vs. actual content). Recommended next
step, if this is picked up: **one LLM call per paper**, prompted the same way the
hand labels here were built (band + one-line reason), rather than expanding the
keyword lists — this sample suggests keyword expansion has a low ceiling, since the
disagreements are about context and framing, not missing vocabulary.